# Geophysical Waveform Inversion: Kaggle Pipeline

Run the cells from top to bottom. The notebook downloads the GitHub repository when needed, validates the environment and data, computes statistics, performs a one-epoch preflight, and provides separate cells for formal training, inference, and submission validation.

In [ ]:
# Kaggle setup: use the current repository or clone it from GitHub.
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/fancyleo/Geophysical-Waveform-Inversion.git"
CURRENT_DIR = Path.cwd()
REPO_DIR = next((path for path in [CURRENT_DIR, *CURRENT_DIR.parents] if (path / "working_space" / "train.py").exists()), CURRENT_DIR / "Geophysical-Waveform-Inversion")
if not (REPO_DIR / "working_space" / "train.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# Kaggle mounts competition data separately from the code repository.
KAGGLE_DATA_ROOT = Path("/kaggle/input/competitions/waveform-inversion")
if not (KAGGLE_DATA_ROOT / "train_samples").is_dir():
    raise FileNotFoundError(f"Training data not found: {KAGGLE_DATA_ROOT / 'train_samples'}")
if not (KAGGLE_DATA_ROOT / "test").is_dir():
    raise FileNotFoundError(f"Test data not found: {KAGGLE_DATA_ROOT / 'test'}")

os.environ["WAVEFORM_DATA_ROOT"] = str(KAGGLE_DATA_ROOT)
os.environ["WAVEFORM_OUTPUT_ROOT"] = "/kaggle/working"
# Reduce CUDA allocator fragmentation during large activation allocations.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Disable tqdm bars in training subprocesses to avoid flooding the cell output buffer.
os.environ["TQDM_DISABLE"] = "1"
sys.path.insert(0, str(REPO_DIR / "working_space"))
print(f"Repository: {REPO_DIR}")
print(f"Input root: {os.environ['WAVEFORM_DATA_ROOT']}")
print(f"Output root: {os.environ['WAVEFORM_OUTPUT_ROOT']}")
print(f"CUDA allocator: {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
print(f"tqdm disabled: {os.environ['TQDM_DISABLE']}")

In [ ]:
# Copy the repository's working_space contents into /kaggle/working.
import shutil

WORKING_SPACE_DIR = REPO_DIR / "working_space"
KAGGLE_WORKING_DIR = Path("/kaggle/working")
KAGGLE_WORKING_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    WORKING_SPACE_DIR,
    KAGGLE_WORKING_DIR,
    dirs_exist_ok=True,
)
print(f"Copied: {WORKING_SPACE_DIR}")
print(f"Destination: {KAGGLE_WORKING_DIR}")

In [ ]:
import importlib
import torch

for package_name in ["numpy", "matplotlib", "sklearn", "tqdm", "torch"]:
    module = importlib.import_module(package_name)
    print(f"{package_name}: {getattr(module, '__version__', 'available')}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from config import Cfg, select_families
from train import find_pairs

print(f"Input root: {Cfg.input_root}")
print(f"Training data: {Cfg.train_data_dir}")
print(f"Test data: {Cfg.test_data_dir}")
print(f"Training data exists: {Cfg.train_data_dir.is_dir()}")
print(f"Test data exists: {Cfg.test_data_dir.is_dir()}")

families = select_families("all")
pairs = find_pairs(str(Cfg.train_data_dir), families)
print(f"Families: {len(families)}; paired files: {len(pairs)}")
assert pairs, "No training pairs were found."

In [ ]:
subprocess.run([sys.executable, str(REPO_DIR / "working_space" / "test_unet.py")], check=True)
subprocess.run([sys.executable, str(REPO_DIR / "working_space" / "smoke_test.py")], check=True)

In [ ]:
# Run one epoch before starting the formal training job.
# Use the same conservative batch size as the formal T4 x2 run.
TRAIN_DIR = Cfg.train_data_dir
OUTPUT_DIR = Cfg.output_dir
BATCH_SIZE = 2
subprocess.run([
    sys.executable, str(REPO_DIR / "working_space" / "train.py"),
    "--data_dir", str(TRAIN_DIR), "--out_dir", str(OUTPUT_DIR),
    "--family", "all", "--epochs", "1",
    "--batch_size", str(BATCH_SIZE),
    "--parallel_mode", "data_parallel",
    "--log_memory",
], check=True)

In [ ]:
# Run this cell for the formal training job.
# T4 cards have 16 GB each; keep the global batch size at 2 for this U-Net.
EPOCHS = 30
BATCH_SIZE = 2
subprocess.run([
    sys.executable, str(REPO_DIR / "working_space" / "train.py"),
    "--data_dir", str(TRAIN_DIR), "--out_dir", str(OUTPUT_DIR),
    "--family", "all", "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--parallel_mode", "data_parallel",
    "--log_memory",
], check=True)

In [ ]:
# Run this cell for the formal training job.
# T4 cards have 16 GB each; keep the global batch size at 4 for this U-Net.
EPOCHS = 30
BATCH_SIZE = 4
subprocess.run([
    sys.executable, str(REPO_DIR / "working_space" / "train.py"),
    "--data_dir", str(TRAIN_DIR), "--out_dir", str(OUTPUT_DIR),
    "--family", "all", "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--parallel_mode", "data_parallel",
], check=True)

In [ ]:
run_dirs = sorted(OUTPUT_DIR.glob("**/model_*/"), key=lambda path: path.stat().st_mtime)
assert run_dirs, "No training run directory was found."
latest_run = run_dirs[-1]
checkpoint = latest_run / "best_unet.pth"
assert checkpoint.exists(), f"Checkpoint not found: {checkpoint}"
submission_path = OUTPUT_DIR / "submission.csv"
print(f"Using checkpoint: {checkpoint}")
subprocess.run([
    sys.executable, str(REPO_DIR / "working_space" / "infer.py"),
    "--ckpt", str(checkpoint), "--test_dir", str(Cfg.test_data_dir),
    "--out", str(submission_path), "--batch_size", str(Cfg.infer_batch_size),
], check=True)

In [ ]:
import pandas as pd
submission = pd.read_csv(submission_path)
expected_columns = 1 + len(range(Cfg.submission_x_start, Cfg.submission_x_stop, Cfg.submission_x_step))
print(f"Submission shape: {submission.shape}")
print(f"Missing values: {int(submission.isna().sum().sum())}")
assert submission.shape[1] == expected_columns
assert submission.isna().sum().sum() == 0
print("Submission validation passed.")